# Classification

## Initial Classification

### Imports

In [10]:
import pandas as pd
import numpy as np
import category_encoders as ce
from category_encoders.binary import BinaryEncoder

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score, classification_report
from sklearn.feature_selection import SelectFromModel
from sklearn.decomposition import PCA

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

In [11]:
data = pd.read_csv('/Users/ashay/Desktop/DREXEL/SPRING 26/DSCI_631/project/sleep_health_dataset.csv')

data.head()

,person_id,age,gender,occupation,bmi,country,sleep_duration_hrs,sleep_quality_score,rem_percentage,deep_sleep_percentage,...,heart_rate_resting_bpm,sleep_aid_used,shift_work,room_temperature_celsius,weekend_sleep_diff_hrs,season,day_type,cognitive_performance_score,sleep_disorder_risk,felt_rested
0,1,29,Female,Driver,25.7,Japan,6.19,6.6,22.5,19.3,...,63,0,0,20.1,1.84,Autumn,Weekday,73.4,Healthy,0
1,2,55,Female,Software Engineer,22.0,USA,8.32,6.9,26.9,14.9,...,52,1,0,18.0,0.13,Winter,Weekend,99.4,Healthy,1
2,3,42,Male,Nurse,25.0,India,3.74,1.0,20.2,16.2,...,72,0,1,17.9,1.67,Spring,Weekend,2.5,Severe,0
3,4,37,Female,Student,29.5,India,6.79,6.4,17.7,17.7,...,71,0,0,19.1,2.37,Summer,Weekend,67.8,Healthy,0
4,5,23,Male,Lawyer,23.6,Spain,5.02,3.2,23.3,18.3,...,71,0,0,19.7,1.26,Summer,Weekday,38.1,Mild,0


### Preprocessing

These columns were dropped as they added noise or were at risk of leaking information.
- person_id: row index
- felt_rested: marked as an outcome of sleep quality, could cause data leakage
- cognitive _performance_score: also an outcome of sleep quality, risk of data leakage

In [12]:
drop_class = ['person_id', 'felt_rested', 'cognitive_performance_score']

class_df = data.drop(columns= drop_class)

print(class_df.shape)

(100000, 29)


In [13]:
num_cols = ['age', 'bmi', 'sleep_duration_hrs', 'sleep_quality_score', 'sleep_latency_mins', 'wake_episodes_per_night', 
            'rem_percentage', 'caffeine_mg_before_bed', 'alcohol_units_before_bed', 'screen_time_before_bed_mins', 
            'exercise_day', 'steps_that_day', 'nap_duration_mins', 'heart_rate_resting_bpm', 'room_temperature_celsius',
            'deep_sleep_percentage', 'stress_score', 'weekend_sleep_diff_hrs']

binary_cols = ['gender', 'day_type', 'occupation', 'mental_health_condition', 'season', 'country', 'chronotype']

In [14]:
# encode target
target_map = {'Healthy': 0, 'Mild': 1, 'Moderate': 2, 'Severe': 3}
class_df['sleep_disorder_risk'] = class_df['sleep_disorder_risk'].map(target_map)

### Train / Test Split - Stratified

Due to the class imbalance identified in EDA, a stratified train/test split was used to ensure the proportion of each class is preserved in both the training and test sets.

In [15]:
X = class_df.drop(columns=['sleep_disorder_risk'])
y = class_df['sleep_disorder_risk']


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training Set Size: {X_train.shape}\nTest Set Size: {X_test.shape}')

Training Set Size: (80000, 28)
Test Set Size: (20000, 28)


### Algorithm Pipelines

A shared preprocessor was built using a ColumnTransformer that applies StandardScaler to numeric features and BinaryEncoder to categorical features. Each model was then put in a Pipeline to ensure all preprocessing steps are applied consistently and without data leakage.

All three pipelines follow the same structure:
1. **Preprocessor:** scales numeric features and binary encodes categoricals
2. **SMOTE:** applied after preprocessing to oversample minority classes on training data only
3. **Classifier:** 

The KNN pipeline includes an additional PCA step between SMOTE and the classifier to reduce dimensionality. Since KNN relies on distance calculations.

In [16]:
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), num_cols),
    ('binary', BinaryEncoder(), binary_cols)
])

lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

knn_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('pca', PCA(n_components=0.95)),
    ('classifier', KNeighborsClassifier())
])

rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('classifier', RandomForestClassifier(random_state=42))
])

### Parameter Tuning

RandomizedSearchCV was used to tune each model with 3-fold cross validation on the training set. Macro F1 was chosen as the scoring metric to ensure hyperparameter selection optimized performance equally across all four classes, preventing the search from favoring parameters that simply predict the majority classes well.

Each model searched over the following parameters:

- **Logistic Regression**: regularization strength C and solver
- **KNN**: number of neighbors n_neighbors, weighting scheme weights, and PCA variance n_components
- **Random Forest**: number of trees n_estimators, tree depth max_depth, minimum samples to split a node min_samples_split, and number of features considered at each split max_features

RandomizedSearchCV was chosen over GridSearchCV to improve efficiency.

In [17]:
param_grids = {
    'Logistic Regression': (lr_pipeline, {
        'classifier__C': [0.01, 0.1, 1, 10],
        'classifier__solver': ['lbfgs', 'saga']
    }),
    'KNN': (knn_pipeline, {
        'classifier__n_neighbors': [3, 5, 7, 11],
        'classifier__weights': ['uniform', 'distance'],
        'pca__n_components': [0.90, 0.95, 0.99]
    }),
    'Random Forest': (rf_pipeline, {
        'classifier__n_estimators': [100, 200, 300],
        'classifier__max_depth': [None, 10, 20, 30],
        'classifier__min_samples_split': [2, 5, 10],
        'classifier__max_features': ['sqrt', 'log2']
    })
}

best_models = {}

for name, (pipeline, params) in param_grids.items():
    print(f"Tuning {name}...")
    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=params,
        n_iter=10,
        scoring='f1_macro',
        cv=3,
        random_state=42,
        n_jobs=-1,
    )
    search.fit(X_train, y_train)
    best_models[name] = search.best_estimator_
    print(f"best params: {search.best_params_}")
    print(f"best cv F1: {round(search.best_score_, 4)}\n")

Tuning Logistic Regression...


/opt/anaconda3/lib/python3.12/site-packages/sklearn/model_selection/_search.py:324: UserWarning: The total space of parameters 8 is smaller than n_iter=10. Running 8 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/anaconda3/lib/python3.12/site-packages/joblib/externals/loky/backend/popen_loky_posix.py", line 180, in <module>
    exitcode = process_obj._bootstrap()
  File "/opt/anacon

best params: {'classifier__solver': 'lbfgs', 'classifier__C': 1}
best cv F1: 0.7167

Tuning KNN...
best params: {'pca__n_components': 0.99, 'classifier__weights': 'uniform', 'classifier__n_neighbors': 5}
best cv F1: 0.5464

Tuning Random Forest...


/opt/anaconda3/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/anaconda3/lib/python3.12/site-packages/joblib/externals/loky/backend/popen_loky_posix.py", line 180, in <module>
    exitcode = process_obj._bootstrap()
  Fil

best params: {'classifier__n_estimators': 300, 'classifier__min_samples_split': 10, 'classifier__max_features': 'sqrt', 'classifier__max_depth': 30}
best cv F1: 0.8021



### Results

In [18]:
results = {}

for name, model in best_models.items():
    y_pred = model.predict(X_test)

    results[name] = {
        'accuracy': round(accuracy_score(y_test, y_pred), 4),
        'recall':   round(recall_score(y_test, y_pred, average='macro'), 4),
        'f1':       round(f1_score(y_test, y_pred, average='macro'), 4)
    }

    print(f"\n{name}")
    print(classification_report(y_test, y_pred,
          target_names=['Healthy', 'Mild', 'Moderate', 'Severe']))

results_df = pd.DataFrame(results).T
results_df


Logistic Regression
              precision    recall  f1-score   support

     Healthy       0.92      0.88      0.90     10831
        Mild       0.75      0.71      0.73      6696
    Moderate       0.44      0.63      0.52      1660
      Severe       0.65      0.78      0.71       813

    accuracy                           0.80     20000
   macro avg       0.69      0.75      0.71     20000
weighted avg       0.81      0.80      0.80     20000


KNN
              precision    recall  f1-score   support

     Healthy       0.86      0.79      0.82     10831
        Mild       0.57      0.52      0.54      6696
    Moderate       0.26      0.44      0.33      1660
      Severe       0.38      0.53      0.44       813

    accuracy                           0.66     20000
   macro avg       0.52      0.57      0.53     20000
weighted avg       0.69      0.66      0.67     20000


Random Forest
              precision    recall  f1-score   support

     Healthy       0.96      0.96 

,accuracy,recall,f1
Logistic Regression,0.7973,0.7496,0.7150
KNN,0.6589,0.5678,0.5329
Random Forest,0.8990,0.7904,0.8063


All three models were evaluated on the held-out test set of 20,000 records. Accuracy, macro recall, and macro F1 were used as evaluation metrics. Macro averaging was chosen to ensure each class is weighted equally regardless of size.

**Logistic Regression** achieved 80% accuracy with a macro F1 of 0.72. Healthy and Mild performed strongest at 0.90 and 0.73 F1 respectively. Moderate was the hardest class at 0.52 F1, with low precision of 0.44 indicating frequent misclassification of other classes as Moderate. Severe showed high recall of 0.78 but lower precision of 0.65, meaning the model catches most Severe cases but with more false alarms.

**KNN** achieved 66% accuracy with a macro F1 of 0.53, the weakest performance across all metrics. Healthy was reasonable at 0.82 F1 but Mild dropped significantly to 0.54. Moderate had poor performance at 0.33 F1 with a precision of 0.26. Both Severe precision and recall were weak, confirming that distance based methods struggle with high dimensional imbalanced data even with PCA applied.

**Random Forest** achieved 90% accuracy with a macro F1 of 0.81, the strongest performance across all metrics. Healthy and Mild were  at 0.96 and 0.88 F1. Moderate remained the hardest class at 0.64 F1, consistent across all three models. Severe showed high precision of 0.84 but lower recall of 0.66, meaning the model is conservative.

Random Forest's performance confirms that there are non-linear relationships in this dataset. The consistent difficulty with Moderate across all models suggests the boundary between Mild and Moderate risk is inherently ambiguous in the data.

## Hierarchical Classification
A two-stage classification approach where Stage 1 separates Healthy from 
Non-Healthy, and Stage 2 classifies Non-Healthy records into Mild, Moderate, 
or Severe. This approach aims to improve performance on minority classes by 
breaking the problem into simpler subproblems.

In [20]:
# step 1
y_train_binary = y_train.map({0: 0, 1: 1, 2: 1, 3: 1})
y_test_binary = y_test.map({0: 0, 1: 1, 2: 1, 3: 1})

# step 2
non_healthy_mask = y_train != 0
X_train_nh = X_train[non_healthy_mask]
y_train_nh = y_train[non_healthy_mask]  

In [21]:
binary_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('classifier', RandomForestClassifier(random_state=42))
])

binary_params = {
    'classifier__n_estimators': [100, 200, 300],
    'classifier__max_depth': [None, 10, 20, 30],
    'classifier__min_samples_split': [2, 5, 10],
    'classifier__max_features': ['sqrt', 'log2']
}

binary_search = RandomizedSearchCV(
    estimator=binary_pipeline,
    param_distributions=binary_params,
    n_iter=10,
    scoring='f1_macro',
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

binary_search.fit(X_train, y_train_binary)
print("stage 1 best params:", binary_search.best_params_)
print("stage 1 cv F1:", round(binary_search.best_score_, 4))

Fitting 3 folds for each of 10 candidates, totalling 30 fits



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/anaconda3/lib/python3.12/site-packages/joblib/externals/loky/backend/popen_loky_posix.py", line 180, in <module>
    exitcode = process_obj._bootstrap()
  File "/opt/anaconda3/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/opt/anaconda3/lib/python3.12/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/opt/anaconda

stage 1 best params: {'classifier__n_estimators': 200, 'classifier__min_samples_split': 5, 'classifier__max_features': 'sqrt', 'classifier__max_depth': None}
stage 1 cv F1: 0.9592


In [22]:
multiclass_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('classifier', RandomForestClassifier(random_state=42))
])

multiclass_search = RandomizedSearchCV(
    estimator=multiclass_pipeline,
    param_distributions=binary_params, 
    n_iter=10,
    scoring='f1_macro',
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

multiclass_search.fit(X_train_nh, y_train_nh)
print("stage 2 best params:", multiclass_search.best_params_)
print("stage 2 cv F1:", round(multiclass_search.best_score_, 4))

Fitting 3 folds for each of 10 candidates, totalling 30 fits
stage 2 best params: {'classifier__n_estimators': 300, 'classifier__min_samples_split': 10, 'classifier__max_features': 'sqrt', 'classifier__max_depth': 30}
stage 2 cv F1: 0.7647


In [23]:
import numpy as np

stage1_model = binary_search.best_estimator_
y_pred_binary = stage1_model.predict(X_test)

stage2_model = multiclass_search.best_estimator_
non_healthy_idx = y_pred_binary == 1

y_pred_final = y_pred_binary.copy()
y_pred_final[non_healthy_idx] = stage2_model.predict(X_test[non_healthy_idx])

print(classification_report(y_test, y_pred_final,
      target_names=['Healthy', 'Mild', 'Moderate', 'Severe']))

print({
    'accuracy': round(accuracy_score(y_test, y_pred_final), 4),
    'recall':   round(recall_score(y_test, y_pred_final, average='macro'), 4),
    'f1':       round(f1_score(y_test, y_pred_final, average='macro'), 4)
})

              precision    recall  f1-score   support

     Healthy       0.96      0.97      0.97     10831
        Mild       0.87      0.89      0.88      6696
    Moderate       0.64      0.61      0.63      1660
      Severe       0.82      0.69      0.75       813

    accuracy                           0.90     20000
   macro avg       0.82      0.79      0.81     20000
weighted avg       0.90      0.90      0.90     20000

{'accuracy': 0.9014, 'recall': 0.7903, 'f1': 0.8059}


- **Stage 1:** (Healthy vs Non-Healthy) achieved a cross validation F1 of 0.96, reflecting that the boundary between Healthy and Non-Healthy sleepers is well defined and easily separable.
- **Stage 2:** (Mild vs Moderate vs Severe) achieved a cross validation F1 of 0.76, confirming that differentiating between severity levels within the non-healthy group remains challenging.

On the test set the hierarchical approach achieved 90% accuracy and a macro F1 of 0.81, close to identical to the standard Random Forest. Healthy improved slightly to 0.97 F1, however Moderate dropped marginally to 0.63 and Severe 
remained unchanged at 0.75.

The hierarchical approach did not meaningfully improve over the single model Random Forest. This suggests that the difficulty in classifying Moderate cases stems from ambiguity between Mild and Moderate risk levels rather than confusion with the Healthy class. Breaking the problem into two stages did not resolve this boundary, indicating that further improvement would require richer 
features or more advanced modeling techniques rather than a structural change to the classification approach.